# Workspace Inventory – Analysis Queries

Run these against the `workspace_inventory_snapshot` Delta table to understand your workspace health.

Each query answers a specific governance question.

## Setup

In [1]:
%%sql
-- Use the latest snapshot only (in case you have multiple runs)
CREATE OR REPLACE TEMP VIEW inventory AS
SELECT *
FROM workspace_inventory_snapshot
WHERE snapshot_id = (
    SELECT snapshot_id 
    FROM workspace_inventory_snapshot 
    ORDER BY snapshot_time_utc DESC 
    LIMIT 1
)

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

---
## Q1. Workspace overview — what do we have?

The first thing to understand: how many items of each type exist, and what's the overall health.

In [2]:
%%sql
-- Item count by type, with governance flags per type
SELECT 
    type,
    COUNT(*) AS total_items,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    SUM(CAST(is_unused_artifact AS INT)) AS unused_items,
    SUM(CAST(has_missing_owner AS INT)) AS no_owner,
    SUM(CAST(is_orphaned_model AS INT) + CAST(is_orphaned_endpoint AS INT)) AS orphaned,
    ROUND(AVG(CAST(cleanup_candidate_score AS DOUBLE)), 1) AS avg_cleanup_score
FROM inventory
GROUP BY type
ORDER BY total_items DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 3, Finished, Available, Finished, False)

<Spark SQL result set with 27 rows and 7 fields>

---
## Q2. Who owns what? — ownership distribution

Identify the top creators and find items with no owner (system-generated or orphaned).

In [3]:
%%sql
-- Top 15 creators by item count, with their stale %
SELECT 
    created_by,
    COUNT(*) AS items_owned,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    ROUND(SUM(CAST(is_stale AS INT)) * 100.0 / COUNT(*), 1) AS stale_pct,
    ROUND(AVG(CAST(days_since_modified AS DOUBLE)), 0) AS avg_days_since_modified
FROM inventory
WHERE created_by IS NOT NULL AND created_by != '' AND created_by != 'None'
GROUP BY created_by
ORDER BY items_owned DESC
LIMIT 15

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 4, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 5 fields>

---
## Q3. What's actually being used? — usage vs. modification

The most important governance question: which items have genuine usage (from Activity Events), which are only modified, and which are completely dormant.

In [4]:
%%sql
-- Usage categories: actively used, modified-only, completely dormant
SELECT 
    CASE 
        WHEN last_used_date IS NOT NULL AND last_used_date != '' AND last_used_date != 'None'
            THEN 'Actively used (has access events)'
        WHEN CAST(days_since_modified AS INT) <= 90 
            THEN 'Recently modified (no access data)'
        WHEN CAST(days_since_modified AS INT) <= 180 
            THEN 'Modified 90-180 days ago'
        ELSE 'Dormant (>180 days no activity)'
    END AS usage_category,
    COUNT(*) AS item_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM inventory), 1) AS pct_of_total
FROM inventory
GROUP BY 1
ORDER BY item_count DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 5, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 3 fields>

---
## Q4. Most accessed items — what's actually valuable?

Items with the highest access counts are the ones you must protect during any cleanup.

In [5]:
%%sql
-- Top 20 most accessed items in the last 30 days
SELECT 
    name,
    type,
    created_by,
    CAST(access_count_30d AS INT) AS access_count_30d,
    CAST(unique_users_30d AS INT) AS unique_users_30d,
    CAST(days_since_last_used AS INT) AS days_since_last_used,
    CAST(days_since_modified AS INT) AS days_since_modified
FROM inventory
WHERE access_count_30d IS NOT NULL AND access_count_30d != '' AND access_count_30d != 'None'
ORDER BY CAST(access_count_30d AS INT) DESC
LIMIT 20

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 6, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 7 fields>

---
## Q5. Stale items breakdown — what's sitting idle?

Breaks down stale items by type and age band to prioritize cleanup.

In [6]:
%%sql
-- Stale items by type and age band
SELECT 
    type,
    CASE 
        WHEN CAST(days_since_modified AS INT) > 365 THEN '5. >1 year'
        WHEN CAST(days_since_modified AS INT) > 270 THEN '4. 270-365 days'
        WHEN CAST(days_since_modified AS INT) > 180 THEN '3. 180-270 days'
        WHEN CAST(days_since_modified AS INT) > 90  THEN '2. 90-180 days'
        ELSE '1. <90 days'
    END AS age_band,
    COUNT(*) AS item_count
FROM inventory
WHERE CAST(is_stale AS INT) = 1
GROUP BY type, 2
ORDER BY type, age_band

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 7, Finished, Available, Finished, False)

<Spark SQL result set with 44 rows and 3 fields>

---
## Q6. Top cleanup candidates — what should we review first?

Items with the highest composite cleanup scores. These combine staleness, unused status, missing owner, orphan status, and age.

In [7]:
%%sql
-- Top 30 cleanup candidates with all governance flags
SELECT 
    name,
    type,
    created_by,
    CAST(cleanup_candidate_score AS INT) AS score,
    CAST(is_stale AS INT) AS stale,
    CAST(is_unused_artifact AS INT) AS unused,
    CAST(has_missing_owner AS INT) AS no_owner,
    CAST(is_orphaned_model AS INT) AS orphan_model,
    CAST(is_orphaned_endpoint AS INT) AS orphan_ep,
    CAST(is_duplicate_name AS INT) AS duplicate,
    CAST(days_since_modified AS INT) AS days_mod,
    CAST(days_since_last_used AS INT) AS days_used
FROM inventory
WHERE CAST(cleanup_candidate_score AS INT) >= 30
ORDER BY CAST(cleanup_candidate_score AS INT) DESC
LIMIT 30

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 8, Finished, Available, Finished, False)

<Spark SQL result set with 30 rows and 12 fields>

---
## Q7. Orphaned items — what has lost its parent or purpose?

Semantic models with no report, SQL endpoints with no lakehouse/warehouse.

In [8]:
%%sql
-- All orphaned items with context
SELECT 
    name,
    type,
    created_by,
    created_date,
    CAST(days_since_modified AS INT) AS days_since_modified,
    CAST(is_unused_artifact AS INT) AS unused,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(is_orphaned_model AS INT) = 1 
   OR CAST(is_orphaned_endpoint AS INT) = 1
ORDER BY CAST(cleanup_candidate_score AS INT) DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 9, Finished, Available, Finished, False)

<Spark SQL result set with 25 rows and 7 fields>

---
## Q8. Unused Power BI artifacts — confirmed no usage

These items were flagged by `list_unused_artifacts()` as having zero usage in Power BI metrics. Combined with their last accessed date from the same API.

In [9]:
%%sql
-- Unused artifacts with their last known access date
SELECT 
    name,
    type,
    created_by,
    created_date,
    last_used_date,
    CAST(days_since_last_used AS INT) AS days_since_used,
    CAST(days_since_modified AS INT) AS days_since_mod,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(is_unused_artifact AS INT) = 1
ORDER BY CAST(cleanup_candidate_score AS INT) DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 10, Finished, Available, Finished, False)

<Spark SQL result set with 38 rows and 8 fields>

---
## Q9. Duplicate items — same name and type appearing more than once

Could indicate copy-paste development, failed deployments, or test artifacts left behind.

In [10]:
%%sql
-- Duplicate items side by side
SELECT 
    a.name,
    a.type,
    a.id,
    a.created_by,
    a.created_date,
    a.last_modified,
    CAST(a.days_since_modified AS INT) AS days_mod
FROM inventory a
WHERE CAST(a.is_duplicate_name AS INT) = 1
ORDER BY a.name, a.type, a.last_modified DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 11, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 7 fields>

---
## Q10. Items with no owner — who created these?

System-generated items, imported items, or items whose creator was deleted from the tenant.

In [11]:
%%sql
-- Items with missing owner
SELECT 
    name,
    type,
    created_date,
    last_modified,
    CAST(days_since_modified AS INT) AS days_mod,
    CAST(is_stale AS INT) AS stale,
    CAST(cleanup_candidate_score AS INT) AS score
FROM inventory
WHERE CAST(has_missing_owner AS INT) = 1
ORDER BY type, name

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 12, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 7 fields>

---
## Q11. Safe-to-keep list — items that are actively used and should NOT be deleted

The inverse of cleanup candidates. Items with recent usage, active modifications, and known owners.

In [12]:
%%sql
-- Items confirmed as actively used (do NOT delete)
SELECT 
    name,
    type,
    created_by,
    CAST(access_count_30d AS INT) AS access_30d,
    CAST(unique_users_30d AS INT) AS users_30d,
    CAST(days_since_last_used AS INT) AS days_since_used,
    CAST(days_since_modified AS INT) AS days_since_mod
FROM inventory
WHERE last_used_date IS NOT NULL 
  AND last_used_date != '' 
  AND last_used_date != 'None'
  AND CAST(is_stale AS INT) = 0
ORDER BY CAST(access_count_30d AS INT) DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 13, Finished, Available, Finished, False)

<Spark SQL result set with 35 rows and 7 fields>

---
## Q12. Creation timeline — when were items created?

Helps identify bulk-creation events (migrations, workshops, hackathons) and the age profile of the workspace.

In [13]:
%%sql
-- Items created per month (where created_date is available)
SELECT 
    DATE_FORMAT(TO_TIMESTAMP(created_date), 'yyyy-MM') AS created_month,
    COUNT(*) AS items_created,
    COLLECT_SET(type) AS types_created
FROM inventory
WHERE created_date IS NOT NULL AND created_date != '' AND created_date != 'None'
GROUP BY 1
ORDER BY 1

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 14, Finished, Available, Finished, False)

<Spark SQL result set with 22 rows and 3 fields>

---
## Q13. Modified-by analysis — who is actively working in this workspace?

Shows who has been modifying items recently, which helps identify active contributors vs. one-time creators.

In [14]:
%%sql
-- Active modifiers (modified_by from Scanner API)
SELECT 
    modified_by,
    COUNT(*) AS items_modified,
    MIN(CAST(days_since_modified AS INT)) AS most_recent_mod_days,
    ROUND(AVG(CAST(days_since_modified AS INT)), 0) AS avg_days_since_mod
FROM inventory
WHERE modified_by IS NOT NULL AND modified_by != '' AND modified_by != 'None'
GROUP BY modified_by
ORDER BY items_modified DESC

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 15, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 4 fields>

---
## Q14. Governance health scorecard — single-number summary

One query that gives you the overall workspace health at a glance.

In [15]:
%%sql
-- Workspace governance scorecard
SELECT
    COUNT(*) AS total_items,
    SUM(CASE WHEN CAST(is_stale AS INT) = 0 THEN 1 ELSE 0 END) AS active_items,
    SUM(CAST(is_stale AS INT)) AS stale_items,
    ROUND(SUM(CAST(is_stale AS INT)) * 100.0 / COUNT(*), 1) AS stale_pct,
    SUM(CAST(is_unused_artifact AS INT)) AS unused_artifacts,
    SUM(CAST(has_missing_owner AS INT)) AS no_owner_items,
    SUM(CAST(is_duplicate_name AS INT)) AS duplicate_items,
    SUM(CAST(is_orphaned_model AS INT)) AS orphaned_models,
    SUM(CAST(is_orphaned_endpoint AS INT)) AS orphaned_endpoints,
    SUM(CASE WHEN CAST(cleanup_candidate_score AS INT) >= 50 THEN 1 ELSE 0 END) AS high_risk_items,
    SUM(CASE WHEN CAST(cleanup_candidate_score AS INT) >= 30 AND CAST(cleanup_candidate_score AS INT) < 50 THEN 1 ELSE 0 END) AS medium_risk_items,
    SUM(CASE WHEN last_used_date IS NOT NULL AND last_used_date != '' AND last_used_date != 'None' THEN 1 ELSE 0 END) AS items_with_usage_data,
    ROUND(AVG(CAST(cleanup_candidate_score AS DOUBLE)), 1) AS avg_cleanup_score
FROM inventory

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 13 fields>

---
## Q15. Lakehouse/Warehouse ecosystem — parent-child relationships

Shows each Lakehouse and Warehouse alongside its auto-generated SQL endpoint, and flags any orphaned endpoints.

In [16]:
%%sql
-- Lakehouse/Warehouse with their SQL endpoints
SELECT 
    COALESCE(p.name, e.name) AS item_name,
    p.type AS parent_type,
    CASE WHEN p.id IS NOT NULL THEN 'Yes' ELSE 'MISSING' END AS parent_exists,
    CASE WHEN e.id IS NOT NULL THEN 'Yes' ELSE 'No endpoint' END AS endpoint_exists,
    p.created_by AS parent_owner,
    CAST(p.days_since_modified AS INT) AS parent_days_mod,
    CAST(e.days_since_modified AS INT) AS endpoint_days_mod
FROM (
    SELECT * FROM inventory WHERE type IN ('Lakehouse', 'Warehouse')
) p
FULL OUTER JOIN (
    SELECT * FROM inventory WHERE type = 'SQLEndpoint'
) e ON LOWER(TRIM(p.name)) = LOWER(TRIM(e.name))
ORDER BY item_name

StatementMeta(, 9b98069b-d768-4121-aa54-ae9f2d818db6, 17, Finished, Available, Finished, False)

<Spark SQL result set with 54 rows and 7 fields>

In [3]:
# Test 3: Fabric-native notification (notebookutils)
try:
    # Check if sendEmail exists
    print(dir(notebookutils.notification))
except Exception as e:
    print(f"notebookutils.notification: {e}")

try:
    # Some Fabric versions have this
    print(dir(notebookutils))
except Exception as e:
    print(f"notebookutils: {e}")

StatementMeta(, 2f40174e-cea7-4e35-82e4-c3ff827906de, 5, Finished, Available, Finished, False)

notebookutils.notification: module 'notebookutils' has no attribute 'notification'
['PBIClient', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_log_exception', '_prepared', 'azureML', 'businessEvents', 'cognitiveService', 'common', 'conf', 'connections', 'credentials', 'data', 'dataflow', 'datascience', 'displayHTML', 'env', 'fabricClient', 'fs', 'help', 'ipython', 'ipythoninterpreter', 'lakehouse', 'magic', 'mssparkutils', 'notebook', 'powerquery', 'prepare', 'runtime', 'session', 'udf', 'variableLibrary', 'visualization', 'warehouse', 'workspace']


In [4]:
# Test 4: Check if we can get a Graph token via the PBI token exchange
# The PBI token (which works) might be exchangeable for Graph scope
import requests

pbi_token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {pbi_token}"}

# Try calling Graph API directly with the PBI token (sometimes works in Fabric)
resp = requests.get("https://graph.microsoft.com/v1.0/me", headers=headers)
print(f"Graph API with PBI token: {resp.status_code}")
if resp.status_code == 200:
    print(f"  It works! Authenticated as: {resp.json().get('displayName')}")
    
    # Now test mail permission
    resp2 = requests.post(
        "https://graph.microsoft.com/v1.0/me/messages",
        headers={**headers, "Content-Type": "application/json"},
        json={
            "subject": "Permission test",
            "body": {"contentType": "Text", "content": "Test"},
            "toRecipients": [{"emailAddress": {"address": "test@test.com"}}]
        }
    )
    print(f"  Create draft: {resp2.status_code}")
    if resp2.status_code == 201:
        print("  ✓ MAIL PERMISSION WORKS with PBI token!")
        draft_id = resp2.json().get('id')
        requests.delete(f"https://graph.microsoft.com/v1.0/me/messages/{draft_id}", headers=headers)
    else:
        print(f"  Mail: {resp2.status_code} — {resp2.text[:200]}")
else:
    print(f"  PBI token not valid for Graph: {resp.text[:200]}")

StatementMeta(, 2f40174e-cea7-4e35-82e4-c3ff827906de, 6, Finished, Available, Finished, False)

Graph API with PBI token: 401
  PBI token not valid for Graph: {"error":{"code":"InvalidAuthenticationToken","message":"Access token validation failure. Invalid audience.","innerError":{"date":"2026-08-10T11:27:32","request-id":"9fb849bc-b1ae-4a77-b517-3c944c8d6d


In [5]:
# Test 5: SMTP with explicit EHLO hostname fix
import smtplib
try:
    server = smtplib.SMTP('smtp.office365.com', 587)
    server.ehlo('xebiaglobal.onmicrosoft.com')  # Force a valid domain
    server.starttls()
    server.ehlo('xebiaglobal.onmicrosoft.com')
    print("SMTP with fixed EHLO: ✓ connected")
    print("To send, you would need: server.login('user@domain', 'password_or_app_password')")
    server.quit()
except Exception as e:
    print(f"SMTP with fixed EHLO: {e}")

StatementMeta(, 2f40174e-cea7-4e35-82e4-c3ff827906de, 7, Finished, Available, Finished, False)

SMTP with fixed EHLO: ✓ connected
To send, you would need: server.login('user@domain', 'password_or_app_password')
